In [ ]:
!pip install -q \
    "transformers>=4.48,<5" \
    "sentencepiece>=0.2" \
    "safetensors>=0.4" \
    "accelerate>=1.2"

print("Stage 3 libraries installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Stage 3 libraries installed.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re
import torch

STAGE2_JSON = Path(
    "/content/drive/MyDrive/KidStory-Qwen2.5/"
    "learnguard_stage2_results/"
    "20260728_102306_"
    "honesty_when_a_child_finds_a_lost_wallet_"
    "answer_spans.json"
)

if not STAGE2_JSON.exists():
    raise FileNotFoundError(
        f"Stage 2 file was not found: {STAGE2_JSON}"
    )

with STAGE2_JSON.open(
    "r",
    encoding="utf-8",
) as file:
    stage2_record = json.load(file)

if stage2_record["stage2_status"] != "COMPLETE":
    raise ValueError(
        "Stage 2 has not been completed."
    )

TOPIC = stage2_record["topic"]
AGE_GROUP = stage2_record["age_group"]
STORY = stage2_record["story"]
SELECTED_SPANS = stage2_record["selected_spans"]

print("Stage 2 result loaded successfully.")
print("Topic:", TOPIC)
print("Age group:", AGE_GROUP)
print("Selected spans:", len(SELECTED_SPANS))

print("\nANSWERS FOR QUESTION GENERATION")

for index, span in enumerate(
    SELECTED_SPANS,
    start=1,
):
    print(
        f"{index}. {span['answer']} "
        f"→ expected: "
        f"{span['expected_question_type']}"
    )

Mounted at /content/drive
Stage 2 result loaded successfully.
Topic: honesty when a child finds a lost wallet
Age group: 8–10
Selected spans: 12

ANSWERS FOR QUESTION GENERATION
1. Emily → expected: who
2. the park → expected: where
3. Timmy → expected: who
4. a small leather wallet → expected: what
5. Mr. Johnson → expected: who
6. Emily's mother → expected: what
7. the number → expected: what
8. earlier that day → expected: when
9. the community → expected: where
10. an honest and responsible decision → expected: what
11. a child → expected: what
12. honesty → expected: what


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

QG_MODEL_PATH = Path(
    "/content/drive/MyDrive/QA_Fairytale/"
    "QG_Phase3/"
    "t5base_QG_answeraware_e20_bs64_beam8/"
    "checkpoint-2000_final"
)

required_qg_files = [
    "config.json",
    "model.safetensors",
    "tokenizer_config.json",
]

if not QG_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"QG model was not found: {QG_MODEL_PATH}"
    )

missing_files = [
    filename
    for filename in required_qg_files
    if not (
        QG_MODEL_PATH / filename
    ).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing QG model files: {missing_files}"
    )

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Loading the fine-tuned QG model...")

qg_tokenizer = AutoTokenizer.from_pretrained(
    QG_MODEL_PATH
)

qg_model = (
    AutoModelForSeq2SeqLM.from_pretrained(
        QG_MODEL_PATH
    )
    .to(device)
    .eval()
)

print("QG model loaded successfully.")
print("Device:", device)
print(
    "Model class:",
    qg_model.__class__.__name__,
)

Loading the fine-tuned QG model...
QG model loaded successfully.
Device: cuda
Model class: T5ForConditionalGeneration


In [ ]:
ASK_TAG = "<ask>\n"


def normalize_text(text: str) -> str:
    """Normalize text for duplicate detection."""

    text = (text or "").lower().strip()
    text = re.sub(r"[^a-z0-9 ?!]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def make_qg_input(
    story: str,
    answer: str,
) -> str:
    """
    Format the story and answer exactly as expected
    by the fine-tuned QG model.
    """

    return (
        "<story>\n"
        + story.strip()
        + "\n</story>\n"
        + "<answer>\n"
        + answer.strip()
        + "\n</answer>\n"
        + ASK_TAG
    )


def clean_question(question: str) -> str:
    """Clean common formatting problems in a generated question."""

    question = (
        question
        or ""
    ).strip().replace("\n", " ")

    question = re.sub(
        r"\s+",
        " ",
        question,
    ).strip()

    # Retain only the first question if the model
    # accidentally generates multiple questions.
    if question.count("?") >= 2:
        question = (
            question.split("?", 1)[0].strip()
            + "?"
        )

    if (
        "?" in question
        and not question.endswith("?")
    ):
        question = question[
            : question.find("?") + 1
        ]

    question = re.sub(
        r"\s+\?$",
        "?",
        question,
    )

    if (
        question
        and question[0].isalpha()
    ):
        question = (
            question[0].upper()
            + question[1:]
        )

    return question


@torch.inference_mode()
def generate_candidate_questions(
    story: str,
    answer: str,
    number_of_questions: int = 8,
    max_new_tokens: int = 64,
) -> list[str]:
    """Generate several unique questions for one answer."""

    model_input = make_qg_input(
        story,
        answer,
    )

    encoded_input = qg_tokenizer(
        model_input,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)

    generated_outputs = qg_model.generate(
        **encoded_input,
        do_sample=False,
        num_beams=max(
            8,
            number_of_questions,
        ),
        num_return_sequences=number_of_questions,
        max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
        early_stopping=True,
    )

    decoded_questions = [
        qg_tokenizer.decode(
            output,
            skip_special_tokens=True,
        ).strip()
        for output in generated_outputs
    ]

    unique_questions = []
    seen_questions = set()

    for raw_question in decoded_questions:
        cleaned_question = clean_question(
            raw_question
        )

        normalized_question = normalize_text(
            cleaned_question
        )

        if (
            cleaned_question
            and normalized_question
            not in seen_questions
        ):
            seen_questions.add(
                normalized_question
            )

            unique_questions.append(
                cleaned_question
            )

    return unique_questions


print("Question-generation functions are ready.")

Question-generation functions are ready.


In [ ]:
TEST_ANSWER = "a small leather wallet"
EXPECTED_TYPE = "what"

test_questions = generate_candidate_questions(
    story=STORY,
    answer=TEST_ANSWER,
    number_of_questions=8,
)

print("TEST ANSWER:", TEST_ANSWER)
print("EXPECTED QUESTION TYPE:", EXPECTED_TYPE)
print(
    "UNIQUE QUESTIONS GENERATED:",
    len(test_questions),
)

print("\nCANDIDATE QUESTIONS")

for index, question in enumerate(
    test_questions,
    start=1,
):
    print(f"{index}. {question}")

TEST ANSWER: a small leather wallet
EXPECTED QUESTION TYPE: what
UNIQUE QUESTIONS GENERATED: 7

CANDIDATE QUESTIONS
1. What did Emily find on the ground?
2. What did Emily find shiny on the ground?
3. What was shiny on the ground when Emily picked up something shiny?
4. What was shiny on the ground when Emily picked it up?
5. What did Emily discover on the ground?
6. What did Emily find when she picked it up?
7. What was shiny on the ground and Emily found?


In [ ]:
WH_WORDS = {
    "who", "what", "where",
    "when", "why", "how",
}

AUXILIARY_WORDS = {
    "did", "was", "were", "is", "are",
    "do", "does", "can", "could",
    "will", "would", "have", "has", "had",
}

AMBIGUOUS_WORDS = {
    "it",
    "something",
    "anything",
    "this",
    "that",
}

QUESTION_STOPWORDS = {
    "what", "who", "where", "when", "why", "how",
    "did", "was", "were", "is", "are", "do", "does",
    "the", "a", "an", "on", "in", "at", "to", "from",
    "and", "or", "when", "she", "he", "they",
}


def question_format_pass(
    question: str,
    expected_type: str,
) -> bool:
    """Check the basic structure of a generated question."""

    if not question:
        return False

    question = question.strip()

    if len(question) < 8 or len(question) > 140:
        return False

    if (
        question.count("?") != 1
        or not question.endswith("?")
    ):
        return False

    words = normalize_text(question).split()

    if not words:
        return False

    if words[0] not in WH_WORDS:
        return False

    if (
        expected_type in WH_WORDS
        and words[0] != expected_type
    ):
        return False

    wh_count = sum(
        word in WH_WORDS
        for word in words
    )

    if wh_count >= 2:
        return False

    if not any(
        word in AUXILIARY_WORDS
        for word in words
    ):
        return False

    return True


def question_quality_score(
    question: str,
    expected_type: str,
) -> float:
    """Score clarity and formatting of a candidate question."""

    words = normalize_text(question).split()

    if not words:
        return -100.0

    score = 0.0

    if words[0] == expected_type:
        score += 3.0

    if question.endswith("?"):
        score += 1.0

    word_count = len(words)

    # Prefer concise but complete questions.
    if 5 <= word_count <= 10:
        score += 2.0
    elif 11 <= word_count <= 14:
        score += 1.0
    else:
        score -= 0.5

    ambiguous_count = sum(
        word in AMBIGUOUS_WORDS
        for word in words
    )

    score -= ambiguous_count * 0.6

    # Penalize repeated meaningful words.
    content_words = [
        word
        for word in words
        if word not in QUESTION_STOPWORDS
    ]

    repeated_content_words = (
        len(content_words)
        - len(set(content_words))
    )

    score -= repeated_content_words * 0.8

    # Penalize known awkward constructions.
    normalized_question = normalize_text(
        question
    )

    awkward_patterns = [
        r"\bfind shiny\b",
        r"\band [a-z]+ found\?$",
        r"\bpicked up something shiny\b",
    ]

    if any(
        re.search(pattern, normalized_question)
        for pattern in awkward_patterns
    ):
        score -= 2.0

    # Prefer the shorter question when other
    # characteristics are equal.
    score -= word_count * 0.01

    return round(score, 3)


def rank_candidate_questions(
    questions: list[str],
    expected_type: str,
) -> list[dict]:
    """Filter and rank candidate questions."""

    ranked_questions = []

    for question in questions:

        if not question_format_pass(
            question,
            expected_type,
        ):
            continue

        ranked_questions.append(
            {
                "question": question,
                "quality_score": (
                    question_quality_score(
                        question,
                        expected_type,
                    )
                ),
            }
        )

    ranked_questions.sort(
        key=lambda item: item["quality_score"],
        reverse=True,
    )

    return ranked_questions


print("Question filtering and scoring are ready.")

Question filtering and scoring are ready.


In [ ]:
ranked_test_questions = rank_candidate_questions(
    questions=test_questions,
    expected_type=EXPECTED_TYPE,
)

print("RANKED TEST QUESTIONS")

for index, result in enumerate(
    ranked_test_questions,
    start=1,
):
    print(
        f"{index}. "
        f"[score={result['quality_score']}] "
        f"{result['question']}"
    )

RANKED TEST QUESTIONS
1. [score=5.93] What did Emily find on the ground?
2. [score=5.93] What did Emily discover on the ground?
3. [score=3.92] What did Emily find shiny on the ground?
4. [score=3.91] What was shiny on the ground and Emily found?


In [ ]:
MINIMUM_QUESTION_SCORE = 5.0
QUESTIONS_TO_KEEP_PER_SPAN = 3

stage3_question_records = []
globally_seen_questions = set()

for span_number, span_record in enumerate(
    SELECTED_SPANS,
    start=1,
):
    answer = span_record["answer"]

    expected_type = span_record[
        "expected_question_type"
    ]

    print("=" * 70)
    print(
        f"SPAN {span_number}/{len(SELECTED_SPANS)}"
    )
    print("Answer:", answer)
    print("Expected type:", expected_type)

    generated_questions = (
        generate_candidate_questions(
            story=STORY,
            answer=answer,
            number_of_questions=8,
        )
    )

    ranked_questions = rank_candidate_questions(
        questions=generated_questions,
        expected_type=expected_type,
    )

    accepted_for_span = 0

    for ranked_question in ranked_questions:

        if (
            ranked_question["quality_score"]
            < MINIMUM_QUESTION_SCORE
        ):
            continue

        question = ranked_question["question"]

        normalized_question = normalize_text(
            question
        )

        if normalized_question in globally_seen_questions:
            continue

        globally_seen_questions.add(
            normalized_question
        )

        stage3_question_records.append(
            {
                "answer_span": answer,
                "expected_question_type": expected_type,
                "question": question,
                "question_quality_score": (
                    ranked_question[
                        "quality_score"
                    ]
                ),
                "story_section": span_record[
                    "story_section"
                ],
                "support_sentence_index": span_record[
                    "support_sentence_index"
                ],
                "support_sentence": span_record[
                    "support_sentence"
                ],
                "span_salience_score": span_record[
                    "score"
                ],
            }
        )

        accepted_for_span += 1

        if (
            accepted_for_span
            >= QUESTIONS_TO_KEEP_PER_SPAN
        ):
            break

    print(
        "Generated:",
        len(generated_questions),
    )
    print(
        "Passed Stage 3 filter:",
        accepted_for_span,
    )

print("=" * 70)
print("Stage 3 generation finished.")
print(
    "Total retained question candidates:",
    len(stage3_question_records),
)

SPAN 1/12
Answer: Emily
Expected type: who
Generated: 8
Passed Stage 3 filter: 2
SPAN 2/12
Answer: the park
Expected type: where
Generated: 8
Passed Stage 3 filter: 1
SPAN 3/12
Answer: Timmy
Expected type: who
Generated: 6
Passed Stage 3 filter: 0
SPAN 4/12
Answer: a small leather wallet
Expected type: what
Generated: 7
Passed Stage 3 filter: 2
SPAN 5/12
Answer: Mr. Johnson
Expected type: who
Generated: 6
Passed Stage 3 filter: 0
SPAN 6/12
Answer: Emily's mother
Expected type: what
Generated: 6
Passed Stage 3 filter: 0
SPAN 7/12
Answer: the number
Expected type: what
Generated: 6
Passed Stage 3 filter: 0
SPAN 8/12
Answer: earlier that day
Expected type: when
Generated: 7
Passed Stage 3 filter: 0
SPAN 9/12
Answer: the community
Expected type: where
Generated: 8
Passed Stage 3 filter: 0
SPAN 10/12
Answer: an honest and responsible decision
Expected type: what
Generated: 8
Passed Stage 3 filter: 3
SPAN 11/12
Answer: a child
Expected type: what
Generated: 7
Passed Stage 3 filter: 0
SPAN 12

In [ ]:
import pandas as pd

stage3_questions_df = pd.DataFrame(
    stage3_question_records
)

if stage3_questions_df.empty:
    print(
        "No questions passed the Stage 3 filter."
    )
else:
    stage3_questions_df.insert(
        0,
        "candidate_id",
        [
            f"Q{index:03d}"
            for index in range(
                1,
                len(stage3_questions_df) + 1,
            )
        ],
    )

    print(
        "Retained candidates:",
        len(stage3_questions_df),
    )

    print(
        "Answer spans with at least one question:",
        stage3_questions_df[
            "answer_span"
        ].nunique(),
    )

    display(
        stage3_questions_df[
            [
                "candidate_id",
                "answer_span",
                "expected_question_type",
                "question",
                "question_quality_score",
                "story_section",
            ]
        ]
    )

Retained candidates: 8
Answer spans with at least one question: 4


,candidate_id,answer_span,expected_question_type,question,question_quality_score,story_section
0,Q001,Emily,who,Who was the owner of the wallet?,5.93,beginning
1,Q002,Emily,who,Who was the owner of the lost wallet?,5.92,beginning
2,Q003,the park,where,Where did the man walk through earlier that day?,5.31,beginning
3,Q004,a small leather wallet,what,What did Emily find on the ground?,5.93,beginning
4,Q005,a small leather wallet,what,What did Emily discover on the ground?,5.93,beginning
5,Q006,an honest and responsible decision,what,What did Emily feel proud of?,5.94,ending
6,Q007,an honest and responsible decision,what,What did Emily feel proud of for taking the wa...,5.90,ending
7,Q008,an honest and responsible decision,what,What did Emily feel proud of for returning the...,5.90,ending


In [ ]:
COMPATIBLE_QUESTION_TYPES = {
    "who": {"who", "what"},
    "where": {"where", "what"},
    "when": {"when", "what"},
    "what": {"what"},
}


def question_format_pass_v2(
    question: str,
    expected_type: str,
) -> bool:
    """Check grammar structure while allowing compatible WH forms."""

    if not question:
        return False

    question = question.strip()

    if len(question) < 8 or len(question) > 140:
        return False

    if (
        question.count("?") != 1
        or not question.endswith("?")
    ):
        return False

    words = normalize_text(question).split()

    if not words or words[0] not in WH_WORDS:
        return False

    allowed_types = COMPATIBLE_QUESTION_TYPES.get(
        expected_type,
        {expected_type},
    )

    if words[0] not in allowed_types:
        return False

    wh_count = sum(
        word in WH_WORDS
        for word in words
    )

    if wh_count >= 2:
        return False

    if not any(
        word in AUXILIARY_WORDS
        for word in words
    ):
        return False

    normalized_question = normalize_text(
        question
    )

    rejection_patterns = [
        r"\bfind shiny\b",
        r"\band [a-z]+ found\?$",
        r"\bpicked up something shiny\b",
    ]

    if any(
        re.search(pattern, normalized_question)
        for pattern in rejection_patterns
    ):
        return False

    return True


def question_quality_score_v2(
    question: str,
    expected_type: str,
) -> float:
    """Score a question with a smaller compatible-WH reward."""

    words = normalize_text(question).split()

    if not words:
        return -100.0

    first_word = words[0]
    score = 0.0

    if first_word == expected_type:
        score += 3.0
    else:
        score += 2.5

    if question.endswith("?"):
        score += 1.0

    word_count = len(words)

    if 5 <= word_count <= 10:
        score += 2.0
    elif 11 <= word_count <= 14:
        score += 1.0
    else:
        score -= 0.5

    ambiguous_count = sum(
        word in AMBIGUOUS_WORDS
        for word in words
    )

    score -= ambiguous_count * 0.6

    content_words = [
        word
        for word in words
        if word not in QUESTION_STOPWORDS
    ]

    repeated_count = (
        len(content_words)
        - len(set(content_words))
    )

    score -= repeated_count * 0.8
    score -= word_count * 0.01

    return round(score, 3)


def rank_candidate_questions_v2(
    questions: list[str],
    expected_type: str,
) -> list[dict]:
    """Filter and rank questions using flexible WH matching."""

    results = []

    for question in questions:

        if not question_format_pass_v2(
            question,
            expected_type,
        ):
            continue

        results.append(
            {
                "question": question,
                "quality_score": (
                    question_quality_score_v2(
                        question,
                        expected_type,
                    )
                ),
            }
        )

    results.sort(
        key=lambda item: item["quality_score"],
        reverse=True,
    )

    return results


print("Flexible Stage 3 question filter is ready.")

Flexible Stage 3 question filter is ready.


In [ ]:
MINIMUM_QUESTION_SCORE_V2 = 4.5
QUESTIONS_TO_KEEP_PER_SPAN_V2 = 4

stage3_question_records_v2 = []
globally_seen_questions_v2 = set()

for span_number, span_record in enumerate(
    SELECTED_SPANS,
    start=1,
):
    answer = span_record["answer"]
    expected_type = span_record[
        "expected_question_type"
    ]

    print("=" * 70)
    print(
        f"SPAN {span_number}/{len(SELECTED_SPANS)}"
    )
    print("Answer:", answer)
    print("Expected type:", expected_type)

    generated_questions = (
        generate_candidate_questions(
            story=STORY,
            answer=answer,
            number_of_questions=10,
        )
    )

    ranked_questions = (
        rank_candidate_questions_v2(
            questions=generated_questions,
            expected_type=expected_type,
        )
    )

    accepted_count = 0

    for result in ranked_questions:

        if (
            result["quality_score"]
            < MINIMUM_QUESTION_SCORE_V2
        ):
            continue

        question_key = normalize_text(
            result["question"]
        )

        if question_key in globally_seen_questions_v2:
            continue

        globally_seen_questions_v2.add(
            question_key
        )

        stage3_question_records_v2.append(
            {
                "answer_span": answer,
                "expected_question_type": expected_type,
                "question": result["question"],
                "question_quality_score": result[
                    "quality_score"
                ],
                "story_section": span_record[
                    "story_section"
                ],
                "support_sentence_index": span_record[
                    "support_sentence_index"
                ],
                "support_sentence": span_record[
                    "support_sentence"
                ],
                "span_salience_score": span_record[
                    "score"
                ],
            }
        )

        accepted_count += 1

        if (
            accepted_count
            >= QUESTIONS_TO_KEEP_PER_SPAN_V2
        ):
            break

    print("Retained:", accepted_count)

print("=" * 70)
print(
    "Total V2 retained candidates:",
    len(stage3_question_records_v2),
)

SPAN 1/12
Answer: Emily
Expected type: who
Retained: 2
SPAN 2/12
Answer: the park
Expected type: where
Retained: 3
SPAN 3/12
Answer: Timmy
Expected type: who
Retained: 1
SPAN 4/12
Answer: a small leather wallet
Expected type: what
Retained: 3
SPAN 5/12
Answer: Mr. Johnson
Expected type: who
Retained: 0
SPAN 6/12
Answer: Emily's mother
Expected type: what
Retained: 0
SPAN 7/12
Answer: the number
Expected type: what
Retained: 4
SPAN 8/12
Answer: earlier that day
Expected type: when
Retained: 0
SPAN 9/12
Answer: the community
Expected type: where
Retained: 0
SPAN 10/12
Answer: an honest and responsible decision
Expected type: what
Retained: 4
SPAN 11/12
Answer: a child
Expected type: what
Retained: 0
SPAN 12/12
Answer: honesty
Expected type: what
Retained: 0
Total V2 retained candidates: 17


In [ ]:
stage3_questions_v2_df = pd.DataFrame(
    stage3_question_records_v2
)

print(
    "Answer spans covered:",
    stage3_questions_v2_df[
        "answer_span"
    ].nunique(),
    "out of",
    len(SELECTED_SPANS),
)

for index, row in stage3_questions_v2_df.iterrows():
    print(
        f"{index + 1}. "
        f"Answer: {row['answer_span']}\n"
        f"   Question: {row['question']}\n"
        f"   Score: {row['question_quality_score']}\n"
    )

Answer spans covered: 6 out of 12
1. Answer: Emily
   Question: Who was the owner of the wallet?
   Score: 5.93

2. Answer: Emily
   Question: Who was the owner of the lost wallet?
   Score: 5.92

3. Answer: the park
   Question: Where did Emily and Timmy meet Mr. Johnson?
   Score: 5.92

4. Answer: the park
   Question: Where did the man walk through earlier that day?
   Score: 5.31

5. Answer: the park
   Question: Where did Emily and Timmy meet to find the owner of the wallet?
   Score: 4.87

6. Answer: Timmy
   Question: Who did Emily ask to help her find the owner of the wallet?
   Score: 4.87

7. Answer: a small leather wallet
   Question: What did Emily find on the ground?
   Score: 5.93

8. Answer: a small leather wallet
   Question: What did Emily discover on the ground?
   Score: 5.93

9. Answer: a small leather wallet
   Question: What did Emily find on a sunny day in the park?
   Score: 4.89

10. Answer: the number
   Question: What did Emily and Timmy find on the business 

In [ ]:
from datetime import datetime, timezone

STAGE3_RESULTS_DIR = Path(
    "/content/drive/MyDrive/KidStory-Qwen2.5/"
    "learnguard_stage3_results"
)

STAGE3_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

safe_topic = re.sub(
    r"[^a-z0-9]+",
    "_",
    TOPIC.lower(),
).strip("_")[:50]

stage3_result_path = STAGE3_RESULTS_DIR / (
    f"{timestamp}_{safe_topic}_"
    "question_candidates.json"
)

stage3_candidates = []

for candidate_number, candidate in enumerate(
    stage3_question_records_v2,
    start=1,
):
    candidate_record = candidate.copy()

    candidate_record["candidate_id"] = (
        f"Q{candidate_number:03d}"
    )

    stage3_candidates.append(
        candidate_record
    )

stage3_record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_stage2_file": str(STAGE2_JSON),
    "qg_model_path": str(QG_MODEL_PATH),
    "topic": TOPIC,
    "age_group": AGE_GROUP,
    "story": STORY,
    "generation_settings": {
        "beam_candidates_per_span": 10,
        "maximum_retained_per_span": 4,
        "minimum_question_quality_score": (
            MINIMUM_QUESTION_SCORE_V2
        ),
        "decoding": "deterministic beam search",
    },
    "selected_answer_span_count": len(
        SELECTED_SPANS
    ),
    "answer_spans_covered": len(
        {
            candidate["answer_span"]
            for candidate in stage3_candidates
        }
    ),
    "retained_candidate_count": len(
        stage3_candidates
    ),
    "question_candidates": stage3_candidates,
    "stage3_status": "COMPLETE",
}

with stage3_result_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        stage3_record,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Stage 3 result saved:")
print(stage3_result_path)

print("\nStage 3 status: COMPLETE")
print(
    "Question candidates:",
    len(stage3_candidates),
)
print(
    "Answer spans covered:",
    stage3_record["answer_spans_covered"],
)

Stage 3 result saved:
/content/drive/MyDrive/KidStory-Qwen2.5/learnguard_stage3_results/20260728_105009_honesty_when_a_child_finds_a_lost_wallet_question_candidates.json

Stage 3 status: COMPLETE
Question candidates: 17
Answer spans covered: 6
